In [24]:
# import all libraries
import pandas as pd
import re
from textblob import TextBlob
from nltk.stem import WordNetLemmatizer, PorterStemmer, LancasterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import CountVectorizer
from autocorrect import Speller

In [25]:
# import and read dataset
df = pd.read_csv('../datasets/Fin_lab-PRProject_dataset.csv')
df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
0,0,77057085,english,Is good. Do play.,0
1,1,77052689,english,AAAAAAAA,0
2,2,77049252,english,Fun game,1
3,3,77049089,english,"Great game, worth every penny!",0
4,4,35101272,english,Like,0


In [26]:
# check the dataset null/nan values
df[df['review'].isnull()]

,Unnamed: 0,recommendationid,language,review,Reaction
181,81,76716062,english,NaN,0
257,57,76570012,english,NaN,1
295,95,76488615,english,NaN,0
418,18,76267675,english,NaN,0
1309,9,74496066,english,NaN,1
...,...,...,...,...,...
38258,58,17084748,english,NaN,1
41587,87,14038822,english,NaN,0
42061,61,13776493,english,NaN,1
42605,5,13528927,english,NaN,1


In [27]:
# check types in 'review' column
df['review'].apply(type).value_counts()

review
<class 'str'>      46656
<class 'float'>       86
Name: count, dtype: int64

In [28]:
# contraction mapping
CONTRACTION_MAP = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "wasn't" : "was not",
    "that's": "that is",
    "you'll" : "you will"
}

In [32]:
# MODULE DECLARATIONS
spellChecker = Speller(lang='en')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# HELPER FUNCTIONS

# tokenize text
def tokenize_text(text):
    word_tokens = word_tokenize(text)
    return word_tokens

# clear rows that has null values
def clear_null_values(text):
    if not isinstance(text, str):
        return ""
    return text

# remove punctuations
def remove_punctuations(text):
    punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*'''
    # remove punctuations from the text
    no_punct = ""
    for char in text:
        if char not in punctuations:
            no_punct = no_punct + char
    return no_punct

# remove repeating characters
def remove_repeating_characters(tokens):
    repeatedPattern = re.compile(r'(\w*)(\w)\2(\w*)')
    matchSubstitution = r'\1\2\3'
    def replace(oldWord):
        if wordnet.synsets(oldWord):
            return oldWord
        newWord = repeatedPattern.sub(matchSubstitution, oldWord)
        return replace(newWord) if newWord != oldWord else newWord
    correctTokens = [replace(word) for word in tokens]
    return correctTokens

# remove contractions
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP):

    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())), flags=re.IGNORECASE|re.DOTALL)

    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = contraction_mapping.get(match) if contraction_mapping.get(match) else contraction_mapping.get(match.lower())
        expanded_contraction = first_char + expanded_contraction[1:]
        return expanded_contraction
    
    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'", "", expanded_text)
    return expanded_text

# remove stopwords
myWordDictionary = []
def remove_stopwords_orig(text):
    data = text
    stopWords = set(stopwords.words('english'))
    words = data

    wordsFiltered = []
    for w in words:
        if w not in stopWords:
            myWordDictionary.append(w)
            wordsFiltered.append(w)
            wordsFiltered.append(' ')

    return "".join(wordsFiltered)



<>:21: SyntaxWarning: invalid escape sequence '\,'
<>:21: SyntaxWarning: invalid escape sequence '\,'
C:\Users\mosqu\AppData\Local\Temp\ipykernel_13640\1338848929.py:21: SyntaxWarning: invalid escape sequence '\,'
  punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*'''


In [ ]:
# CLEAN TEXT PIPELINE
def clean_text_pipeline(text, do_spellcheck=True):
    # 1 - handle nulls
    text = clear_null_values(text)

    # 2 - expand contractions
    text = expand_contractions(text)

    # 3 - remove punctuations
    text = remove_punctuations(text)

    # 4 - turn all text to lowercase
    text = text.lower()

    # 5 - tokenize text
    tokens = tokenize_text(text)

    # 6 - remove repeating characters
    tokens = remove_repeating_characters(tokens)

    # 7 - spell check
    if do_spellcheck:
        tokens = [spellChecker(word) for word in tokens]

    # 8 - remove stopwords
    tokens = remove_stopwords_orig(tokens)

    # 9 - stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # 10 - lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # 11 - join back the cleaned text into a single string
    final_cleaned_text = ' '.join(tokens)

    return final_cleaned_text.strip()
